# MVP-02 — Person detector baseline (YOLO26n)

Train a **baseline person detector** on the VisDrone → person-only YOLO dataset prepared in MVP-01.

| Item | Value |
| --- | --- |
| Model | pretrained `yolo26n.pt` |
| Classes | `0 = person` only |
| Goal | Reproducible baseline (not hyperparameter tuning) |
| Training location | **Google Colab GPU** |

Dataset path:

```text
Google Drive  visdrone_person.zip
        ↓
extract once
        ↓
/content/datasets/visdrone_person/
        ↓
sanity check
        ↓
YOLO training
```

**This notebook does not train until you set `RUN_TRAINING = True`.**

Uniform classification (`our_team` / `known_friendly` / `unknown`) is out of scope for MVP-02.

## 1. Configuration

**What:** All user-editable experiment settings live in one cell.

**Why:** Keeps the run reproducible and easy to change without hunting through the notebook.

Edit this cell, then run the rest top-to-bottom.

In [ ]:
# === EDIT ME (MVP-02 baseline) ===
REPO_URL = "YOUR_GITHUB_REPO_URL"
REPO_BRANCH = "main"
PROJECT_DIR = "/content/uniform-detection"

DRIVE_ROOT = "/content/drive/MyDrive/uniform-detection"

# Upload the MVP-01 processed dataset as a single zip (not a Drive folder tree):
#   DRIVE_ROOT/datasets/visdrone_person.zip
DATASET_ZIP = "/content/drive/MyDrive/uniform-detection/datasets/visdrone_person.zip"
LOCAL_DATASET_DIR = "/content/datasets/visdrone_person"

DRIVE_CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints/detector"
DRIVE_EXPERIMENT_DIR = f"{DRIVE_ROOT}/experiments"

# Extract zip → Colab local disk (skipped if already extracted this runtime)
EXTRACT_DATASET = True

MODEL_NAME = "yolo26n.pt"
IMAGE_SIZE = 640
EPOCHS = 50
BATCH_SIZE = 16
WORKERS = 2
PATIENCE = 10
SEED = 42

EXPERIMENT_NAME = "mvp02_yolo26n_visdrone_person_baseline"

# Project-specific normalized-area bins (NOT COCO thresholds)
SMALL_AREA_THRESHOLD = 0.001
MEDIUM_AREA_THRESHOLD = 0.01
SIZE_RECALL_IOU = 0.5
SIZE_RECALL_MAX_IMAGES = 200  # cap for speed during post-train analysis

INFERENCE_VIZ_COUNT = 12
LATENCY_WARMUP = 5
LATENCY_RUNS = 50

# Safety gate: training runs ONLY when this is True
RUN_TRAINING = False

print("Configuration loaded.")
print("DATASET_ZIP =", DATASET_ZIP)
print("LOCAL_DATASET_DIR =", LOCAL_DATASET_DIR)
print("EXTRACT_DATASET =", EXTRACT_DATASET)
print("RUN_TRAINING =", RUN_TRAINING)
if not RUN_TRAINING:
    print(
        "Training is DISABLED. Set RUN_TRAINING = True in this cell only when "
        "you are ready to start the Colab GPU baseline run."
    )

## 2. Environment / GPU verification

**What:** Print Python / torch / CUDA / Ultralytics versions and GPU name.

**Why:** MVP-02 baseline timing and quality assume a Colab **GPU** runtime. We refuse to start training on CPU by accident.

In [ ]:
import platform
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
GPU_NAME = None
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    print("GPU:", GPU_NAME)
else:
    print(
        "WARNING: No CUDA GPU detected. In Colab: Runtime → Change runtime type → GPU."
    )

try:
    import ultralytics

    ULTRALYTICS_VERSION = getattr(ultralytics, "__version__", "unknown")
    print("ultralytics:", ULTRALYTICS_VERSION)
except ImportError:
    ULTRALYTICS_VERSION = None
    print("ultralytics: not installed yet (install step comes after clone)")

## 3. Mount Google Drive

**What:** Attach your Drive so the notebook can read `visdrone_person.zip` and write checkpoints.

**Why:** The processed dataset stays on Drive as **one zip** (not GitHub, not thousands of Drive files). Colab extracts it once onto local disk; `best.pt` still lands on Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted.")

## 4. Clone or update GitHub repository

**What:** Get the project code (evaluation helpers, requirements, configs).

**Why:** Training orchestration stays in the notebook; reusable logic lives in `src/`.

In [ ]:
from pathlib import Path

project_path = Path(PROJECT_DIR)
if REPO_URL == "YOUR_GITHUB_REPO_URL":
    print(
        "WARNING: REPO_URL is still a placeholder. Set your real GitHub URL in the config cell."
    )

if project_path.exists() and (project_path / ".git").exists():
    %cd {PROJECT_DIR}
    !git fetch origin
    !git checkout {REPO_BRANCH}
    !git pull origin {REPO_BRANCH}
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

print("Project cwd:", Path.cwd())

## 5. Install dependencies

**What:** Install the repo `requirements.txt` (includes Ultralytics).

**Why:** Matches the local development stack so `src.evaluation` imports work.

In [ ]:
!pip install -q -r requirements.txt

import ultralytics
import torch

ULTRALYTICS_VERSION = getattr(ultralytics, "__version__", "unknown")
print("ultralytics:", ULTRALYTICS_VERSION)
print("torch:", torch.__version__)

## 6. Extract dataset zip

**What:** Unzip Drive `visdrone_person.zip` **once** onto Colab local disk at `/content/datasets/visdrone_person/`.

**Why:** Training I/O is much faster on local disk than Drive. A single zip is also simpler to upload than the full YOLO folder tree.

If that local path already has train/val images, unzipping is skipped. Set `EXTRACT_DATASET = True` the first time so the zip is unpacked.

In [ ]:
import shutil
import zipfile
from pathlib import Path

zip_path = Path(DATASET_ZIP)
local_dataset = Path(LOCAL_DATASET_DIR)


def _split_nonempty(root: Path, split: str) -> bool:
    img = root / "images" / split
    lbl = root / "labels" / split
    return img.is_dir() and lbl.is_dir() and any(img.iterdir())


def _dataset_ready(root: Path) -> bool:
    return root.is_dir() and _split_nonempty(root, "train") and _split_nonempty(root, "val")


def _find_yolo_root(start: Path):
    """Locate images/ + labels/ even if the zip nested a visdrone_person/ folder."""
    start = start.resolve()
    queue = [start]
    for _ in range(4):
        next_queue = []
        for node in queue:
            if (node / "images").is_dir() and (node / "labels").is_dir():
                return node
            next_queue.extend(p for p in node.iterdir() if p.is_dir())
        queue = next_queue
    return None


if _dataset_ready(local_dataset):
    DATASET_DIR = local_dataset
    DATASET_MODE = "colab_local_existing"
    print("Using existing extracted dataset:", DATASET_DIR)
elif not EXTRACT_DATASET:
    raise FileNotFoundError(
        f"EXTRACT_DATASET is False and local dataset is not ready:\n  {local_dataset}\n"
        "Set EXTRACT_DATASET = True to unzip from Drive, or extract manually first."
    )
else:
    if not zip_path.is_file():
        raise FileNotFoundError(
            f"Dataset zip not found:\n  {zip_path}\n\n"
            "Upload the MVP-01 processed dataset as a single zip:\n"
            f"  {DRIVE_ROOT}/datasets/visdrone_person.zip\n\n"
            "Zip the visdrone_person folder so it contains images/ and labels/\n"
            "(train + val). Do not re-download VisDrone here — MVP-01 owns preparation."
        )

    extract_tmp = Path("/content/datasets/_visdrone_person_extract")
    if extract_tmp.exists():
        shutil.rmtree(extract_tmp)
    extract_tmp.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {zip_path.name} → local disk (once; may take a few minutes)...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.infolist()
        print("Zip members:", len(members))
        dest = extract_tmp.resolve()
        for info in members:
            target = (dest / info.filename).resolve()
            try:
                target.relative_to(dest)
            except ValueError as exc:
                raise ValueError(f"Unsafe zip path: {info.filename!r}") from exc
        zf.extractall(dest)

    found = _find_yolo_root(extract_tmp)
    if found is None:
        raise FileNotFoundError(
            f"Zip extracted but YOLO layout not found under {extract_tmp}.\n"
            "Expected images/train and labels/train (a top-level visdrone_person/ folder is OK)."
        )

    local_dataset.parent.mkdir(parents=True, exist_ok=True)
    if local_dataset.exists():
        shutil.rmtree(local_dataset)
    if found.resolve() != local_dataset.resolve():
        shutil.move(str(found), str(local_dataset))

    if extract_tmp.exists() and extract_tmp.resolve() != local_dataset.resolve():
        shutil.rmtree(extract_tmp, ignore_errors=True)

    if not _dataset_ready(local_dataset):
        raise FileNotFoundError(
            f"Extract finished but train/val images+labels are missing under {local_dataset}"
        )

    DATASET_DIR = local_dataset
    DATASET_MODE = "zip_extracted"
    print("Extract complete:", DATASET_DIR)

print("DATASET_MODE =", DATASET_MODE)
print("DATASET_DIR  =", DATASET_DIR)

## 7. Validate dataset structure

**What:** Confirm train/val images and labels exist; sample labels for class id `0` only and normalized coords.

**Why:** Fail early before spending GPU hours on a bad path or multi-class leak.

In [ ]:
from src.evaluation.dataset_checks import validate_person_yolo_dataset

sanity = validate_person_yolo_dataset(DATASET_DIR, required_splits=("train", "val"))
for name, counts in sanity.splits.items():
    print(f"{name}: images={counts.num_images} labels={counts.num_labels}")
print("Sampled class IDs:", sanity.label_class_ids)
print("Invalid coordinate lines in sample:", sanity.invalid_coordinate_lines)
print("(Expected ~6471 train / ~548 val images+labels from MVP-01 conversion.)")

TRAIN_IMAGES = sanity.splits["train"].num_images
VAL_IMAGES = sanity.splits["val"].num_images

## 8. Build runtime dataset YAML

**What:** Write an absolute-path Ultralytics YAML for this session.

**Why:** The committed `configs/visdrone_person.yaml` uses relative paths; Colab CWD makes that brittle.

In [ ]:
from pathlib import Path

import yaml

RUNTIME_YAML = Path("/content/visdrone_person_colab.yaml")
data_yaml = {
    "path": str(Path(DATASET_DIR).resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "person"},
}
RUNTIME_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")
print("Wrote", RUNTIME_YAML)
print(RUNTIME_YAML.read_text(encoding="utf-8"))

## 9. Load pretrained YOLO model

**What:** `YOLO(MODEL_NAME)` with default `yolo26n.pt`.

**Why:** A small pretrained nano model is a cheap, standard baseline before comparing larger models later.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
print("Loaded pretrained model:", MODEL_NAME)

## 10. Baseline configuration summary

**What:** Print the frozen experiment settings before any training.

**Why:** Makes the run auditable in the notebook output / later metadata JSON.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_DIR = Path(DRIVE_EXPERIMENT_DIR) / EXPERIMENT_NAME
CHECKPOINT_OUT = Path(DRIVE_CHECKPOINT_DIR) / EXPERIMENT_NAME
RUNS_DIR = EXPERIMENT_DIR / "ultralytics_runs"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_OUT.mkdir(parents=True, exist_ok=True)

BASELINE_SUMMARY = {
    "experiment_name": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "dataset_zip": DATASET_ZIP,
    "dataset_dir": str(DATASET_DIR),
    "dataset_mode": DATASET_MODE,
    "extract_dataset": EXTRACT_DATASET,
    "train_images": TRAIN_IMAGES,
    "val_images": VAL_IMAGES,
    "image_size": IMAGE_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "workers": WORKERS,
    "patience": PATIENCE,
    "seed": SEED,
    "run_training": RUN_TRAINING,
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "ultralytics": ULTRALYTICS_VERSION,
    "note_seed": (
        "SEED is set for Ultralytics; full GPU-level determinism is not guaranteed."
    ),
}
for k, v in BASELINE_SUMMARY.items():
    print(f"{k}: {v}")

## 11. Train

**What:** Ultralytics `model.train(...)` on the person-only dataset.

**Why:** Establish whether YOLO26n can learn VisDrone people at imgsz 640.

Requires `RUN_TRAINING = True` **and** CUDA.

In [ ]:
from pathlib import Path

import torch

train_results = None
BEST_PT = None
LAST_PT = None

if not RUN_TRAINING:
    print(
        "Training skipped: RUN_TRAINING is False.\n"
        "When ready: set RUN_TRAINING = True in the configuration cell, "
        "ensure Colab GPU is enabled, then re-run from the config cell downward."
    )
else:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "RUN_TRAINING=True but CUDA is unavailable. "
            "Enable a Colab GPU runtime (Runtime → Change runtime type → GPU) "
            "and do not train this baseline on CPU by accident."
        )
    train_results = model.train(
        data=str(RUNTIME_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        workers=WORKERS,
        patience=PATIENCE,
        seed=SEED,
        device=0,
        project=str(RUNS_DIR),
        name=EXPERIMENT_NAME,
        exist_ok=True,
    )
    weights_dir = RUNS_DIR / EXPERIMENT_NAME / "weights"
    BEST_PT = weights_dir / "best.pt"
    LAST_PT = weights_dir / "last.pt"
    print("Training finished.")
    print("best.pt:", BEST_PT)
    print("last.pt:", LAST_PT)

## 12. Validate

**What:** Load `best.pt` and run Ultralytics validation; capture Precision, Recall, mAP50, mAP50-95.

**Why:** Overall validation quality for the baseline — separate from small-object analysis.

In [ ]:
from pathlib import Path

from ultralytics import YOLO

from src.evaluation.metrics_io import build_metrics_template, save_json

metrics = build_metrics_template(
    experiment_name=EXPERIMENT_NAME,
    model=MODEL_NAME,
    image_size=IMAGE_SIZE,
    epochs_requested=EPOCHS,
)

if BEST_PT is None or not Path(BEST_PT).is_file():
    # Allow resuming analysis from a previously saved Drive checkpoint
    candidate = Path(DRIVE_CHECKPOINT_DIR) / EXPERIMENT_NAME / "best.pt"
    if candidate.is_file():
        BEST_PT = candidate
        print("Using existing Drive checkpoint:", BEST_PT)
    else:
        print("No best.pt available yet — train first (RUN_TRAINING=True) or place best.pt on Drive.")
        BEST_PT = None

val_results = None
if BEST_PT is not None:
    best_model = YOLO(str(BEST_PT))
    val_results = best_model.val(data=str(RUNTIME_YAML), imgsz=IMAGE_SIZE, split="val")
    box = getattr(val_results, "box", None)
    if box is not None:
        metrics["precision"] = float(box.mp) if getattr(box, "mp", None) is not None else None
        metrics["recall"] = float(box.mr) if getattr(box, "mr", None) is not None else None
        metrics["map50"] = float(box.map50) if getattr(box, "map50", None) is not None else None
        metrics["map50_95"] = float(box.map) if getattr(box, "map", None) is not None else None
    print("Precision:", metrics["precision"])
    print("Recall:", metrics["recall"])
    print("mAP50:", metrics["map50"])
    print("mAP50-95:", metrics["map50_95"])
else:
    print("Validation skipped (no checkpoint).")

## 13. Inspect metrics (including small-object analysis)

**What:**
1. GT normalized-area size distribution on val (project bins, **not** COCO).
2. Detection **recall @ IoU 0.5** per size bin via greedy one-to-one matching.

**Why:** ~89% of VisDrone person boxes are tiny; overall mAP can hide catastrophic small-person failures that matter for aerial/drone use.

In [ ]:
from pathlib import Path

from src.evaluation.label_ops import (
    gt_size_distribution_for_split,
    list_split_images,
    parse_yolo_label_file,
    size_bin_recall_from_predictions,
)
from src.evaluation.size_analysis import Box
from ultralytics import YOLO

gt_dist = gt_size_distribution_for_split(
    Path(DATASET_DIR),
    "val",
    small_area_threshold=SMALL_AREA_THRESHOLD,
    medium_area_threshold=MEDIUM_AREA_THRESHOLD,
)
metrics["gt_size_distribution"] = gt_dist
print("GT size distribution (val, normalized area bins):")
for k, v in gt_dist.items():
    print(f"  {k}: {v}")

size_recall = None
if BEST_PT is not None:
    best_model = YOLO(str(BEST_PT))
    images = list_split_images(Path(DATASET_DIR), "val")[:SIZE_RECALL_MAX_IMAGES]
    image_preds = []
    for img_path in images:
        pred = best_model.predict(source=str(img_path), imgsz=IMAGE_SIZE, verbose=False)[0]
        boxes = []
        if pred.boxes is not None and len(pred.boxes):
            # xywhn: normalized xywh
            xywhn = pred.boxes.xywhn.cpu().tolist()
            for row in xywhn:
                boxes.append(Box(row[0], row[1], row[2], row[3]))
        image_preds.append((img_path, boxes))

    size_recall = size_bin_recall_from_predictions(
        image_preds,
        Path(DATASET_DIR),
        "val",
        iou_threshold=SIZE_RECALL_IOU,
        small_area_threshold=SMALL_AREA_THRESHOLD,
        medium_area_threshold=MEDIUM_AREA_THRESHOLD,
    )
    metrics["size_bin_recall"] = size_recall
    print("\nSize-bin recall @ IoU", SIZE_RECALL_IOU, f"(first {len(images)} val images):")
    for bin_name in ("small", "medium", "large"):
        block = size_recall[bin_name]
        print(
            f"  {bin_name}: recall={block['recall']:.4f} "
            f"(tp={block['tp']} / gt={block['gt']})"
        )
else:
    print("Size-bin recall skipped (no best.pt). GT distribution still saved.")

## 14. Run inference on validation samples

**What:** Predict on ~12 val images and display a few inline; save under the experiment folder.

**Why:** Qualitative check for crowded scenes and tiny people — numbers alone are not enough.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO

from src.evaluation.label_ops import list_split_images, parse_yolo_label_file

pred_dir = EXPERIMENT_DIR / "predictions"
pred_dir.mkdir(parents=True, exist_ok=True)

if BEST_PT is None:
    print("Inference visualization skipped (no best.pt).")
else:
    rng = random.Random(SEED)
    all_val = list_split_images(Path(DATASET_DIR), "val")
    # Prefer a mix: crowded (many GT), sparse (few GT), and mid
    scored = []
    for p in all_val:
        n = len(parse_yolo_label_file(Path(DATASET_DIR) / "labels" / "val" / f"{p.stem}.txt"))
        scored.append((n, p))
    scored.sort(key=lambda t: t[0])
    sparse = [p for n, p in scored[: max(1, len(scored) // 5)]]
    crowded = [p for n, p in scored[-max(1, len(scored) // 5) :]]
    mid = [p for n, p in scored[len(scored) // 3 : 2 * len(scored) // 3]]
    picks = []
    for pool, k in ((sparse, 3), (crowded, 5), (mid, 4)):
        picks.extend(rng.sample(pool, min(k, len(pool))) if pool else [])
    # unique preserve order
    seen = set()
    samples = []
    for p in picks:
        if p not in seen:
            seen.add(p)
            samples.append(p)
    samples = samples[:INFERENCE_VIZ_COUNT]

    best_model = YOLO(str(BEST_PT))
    results = best_model.predict(
        source=[str(p) for p in samples],
        imgsz=IMAGE_SIZE,
        save=True,
        project=str(pred_dir),
        name="val_samples",
        exist_ok=True,
    )
    print("Saved predictions under", pred_dir / "val_samples")

    show_dir = pred_dir / "val_samples"
    shown = 0
    for img_file in sorted(show_dir.glob("*.jpg"))[:4]:
        bgr = cv2.imread(str(img_file))
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(rgb)
        plt.title(img_file.name)
        plt.axis("off")
        plt.show()
        shown += 1
    print("Displayed", shown, "prediction image(s).")

## 15. Save checkpoints to Google Drive

**What:** Copy `best.pt` / `last.pt` and key Ultralytics plots into the experiment + checkpoint folders on Drive.

**Why:** Colab disks are ephemeral; Drive is where the baseline must survive.

In [ ]:
import shutil
from pathlib import Path

run_dir = RUNS_DIR / EXPERIMENT_NAME
weights_dir = run_dir / "weights"
CHECKPOINT_OUT.mkdir(parents=True, exist_ok=True)
(EXPERIMENT_DIR / "artifacts").mkdir(parents=True, exist_ok=True)

copied = []
for name in ("best.pt", "last.pt"):
    src = weights_dir / name
    if src.is_file():
        dst = CHECKPOINT_OUT / name
        shutil.copy2(src, dst)
        copied.append(str(dst))
        print("Copied", src, "->", dst)
        if name == "best.pt":
            BEST_PT = dst
    else:
        print("Missing (train first):", src)

artifact_names = [
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "PR_curve.png",
    "BoxF1_curve.png",
    "F1_curve.png",
    "args.yaml",
]
for name in artifact_names:
    src = run_dir / name
    if src.is_file():
        dst = EXPERIMENT_DIR / "artifacts" / name
        shutil.copy2(src, dst)
        print("Artifact:", dst)

print("Checkpoint folder:", CHECKPOINT_OUT)
print("Files copied:", len(copied))

## 16. Save experiment metrics (+ latency / model size)

**What:** Measure warm-up + timed inference latency, record `best.pt` size / params, write `metrics.json` and `metadata.json`.

**Why:** Baseline must report quality **and** cost (speed, size) for later model comparisons.

In [ ]:
import time
from datetime import datetime, timezone
from pathlib import Path

from ultralytics import YOLO

from src.evaluation.label_ops import list_split_images
from src.evaluation.metrics_io import save_json

if BEST_PT is not None and Path(BEST_PT).is_file():
    best_path = Path(BEST_PT)
    metrics["checkpoint_size_mb"] = round(best_path.stat().st_size / (1024 * 1024), 3)

    best_model = YOLO(str(best_path))
    try:
        metrics["parameter_count"] = int(sum(p.numel() for p in best_model.model.parameters()))
    except Exception as exc:  # noqa: BLE001
        print("Could not count parameters:", exc)

    timing_images = list_split_images(Path(DATASET_DIR), "val")[: max(LATENCY_RUNS, LATENCY_WARMUP)]
    if timing_images:
        # Warm-up (excluded from timing)
        for img in timing_images[:LATENCY_WARMUP]:
            best_model.predict(source=str(img), imgsz=IMAGE_SIZE, verbose=False)

        timed = timing_images[:LATENCY_RUNS]
        t0 = time.perf_counter()
        for img in timed:
            best_model.predict(source=str(img), imgsz=IMAGE_SIZE, verbose=False)
        elapsed = time.perf_counter() - t0
        n = len(timed)
        avg_ms = (elapsed / n) * 1000.0 if n else None
        fps = (n / elapsed) if elapsed > 0 else None
        metrics["avg_latency_ms"] = avg_ms
        metrics["approx_fps"] = fps
        metrics["latency_note"] = (
            "End-to-end Ultralytics predict() per image (includes pre/post as exposed by the API); "
            f"warmup={LATENCY_WARMUP}, runs={n}, imgsz={IMAGE_SIZE}, batch=1, gpu={GPU_NAME}"
        )
        print(f"Average inference latency: {avg_ms:.2f} ms/image")
        print(f"Approx FPS: {fps:.2f}")
        print("Hardware:", GPU_NAME, "| imgsz:", IMAGE_SIZE, "| batch: 1")
else:
    print("Latency / model-size measurement skipped (no best.pt).")

metrics_path = EXPERIMENT_DIR / "metrics.json"
save_json(metrics, metrics_path)
print("Wrote", metrics_path)

metadata = {
    "experiment_name": EXPERIMENT_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model": MODEL_NAME,
    "dataset_zip": DATASET_ZIP,
    "dataset": str(DATASET_DIR),
    "dataset_mode": DATASET_MODE,
    "train_images": TRAIN_IMAGES,
    "val_images": VAL_IMAGES,
    "epochs_requested": EPOCHS,
    "epochs_completed": None,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "patience": PATIENCE,
    "seed": SEED,
    "gpu": GPU_NAME,
    "torch_version": torch.__version__,
    "ultralytics_version": ULTRALYTICS_VERSION,
    "reproducibility_note": (
        "Seed is set where supported; complete GPU determinism is not guaranteed."
    ),
}
# Try to read epochs completed from results.csv if present
results_csv = EXPERIMENT_DIR / "artifacts" / "results.csv"
if not results_csv.is_file():
    results_csv = RUNS_DIR / EXPERIMENT_NAME / "results.csv"
if results_csv.is_file():
    lines = results_csv.read_text(encoding="utf-8", errors="replace").strip().splitlines()
    # header + N epoch rows
    metadata["epochs_completed"] = max(0, len(lines) - 1)

metadata_path = EXPERIMENT_DIR / "metadata.json"
save_json(metadata, metadata_path)
print("Wrote", metadata_path)

## 17. Final experiment summary

**What:** Print paths and headline metrics for the run log / lab notebook.

In [ ]:
print("=" * 60)
print("MVP-02 baseline summary")
print("=" * 60)
print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("RUN_TRAINING:", RUN_TRAINING)
print("Dataset zip:", DATASET_ZIP)
print("Dataset:", DATASET_DIR, f"({DATASET_MODE})")
print("Runtime YAML:", RUNTIME_YAML)
print("Checkpoints:", CHECKPOINT_OUT)
print("Experiment dir:", EXPERIMENT_DIR)
print("Precision:", metrics.get("precision"))
print("Recall:", metrics.get("recall"))
print("mAP50:", metrics.get("map50"))
print("mAP50-95:", metrics.get("map50_95"))
print("Checkpoint MB:", metrics.get("checkpoint_size_mb"))
print("Params:", metrics.get("parameter_count"))
print("Latency ms/img:", metrics.get("avg_latency_ms"))
print("Approx FPS:", metrics.get("approx_fps"))
if metrics.get("size_bin_recall"):
    for bin_name in ("small", "medium", "large"):
        block = metrics["size_bin_recall"][bin_name]
        print(f"{bin_name} recall@IoU{SIZE_RECALL_IOU}: {block['recall']:.4f}")
print("=" * 60)
if not RUN_TRAINING and BEST_PT is None:
    print("Status: READY FOR COLAB TRAINING (set RUN_TRAINING=True on a GPU runtime).")
elif BEST_PT is not None:
    print("Status: checkpoint present — inspect metrics.json on Drive.")
else:
    print("Status: incomplete — train or load a checkpoint.")